In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/industrial-ai-project/MetroPT3(AirCompressor).csv")
df.head()

,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


---
##**Understand the sampling rate of the sensor data**

**Why:** Before any modeling, we must know how frequently readings were recorded, and whether that frequency is consistent (no unexpected gaps).
---

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
time_diffs = df['timestamp'].diff()
print(time_diffs.value_counts())


timestamp
0 days 00:00:10    1337521
0 days 00:00:09     128277
0 days 00:00:12      38321
0 days 00:00:13       7988
0 days 00:00:11       4471
                    ...   
0 days 01:31:44          1
0 days 00:22:24          1
0 days 00:21:53          1
0 days 04:37:19          1
0 days 00:02:35          1
Name: count, Length: 337, dtype: int64


In [ ]:
# Convert the True/False "new segment" flags into a running segment ID
df['segment_id'] = df['new_segment'].cumsum()

# Check: how many rows does each segment have, on average?
print(df['segment_id'].value_counts().describe())

# Investigate Large Time Gaps

In [ ]:
df['time_diff']=time_diffs
big_gaps=df[df['time_diff'] > pd.Timedelta(minutes=5)]
print(big_gaps[['timestamp', 'time_diff']])

                  timestamp       time_diff
4654    2020-02-01 12:53:47 0 days 00:05:07
7114    2020-02-01 23:15:33 0 days 03:35:29
7144    2020-02-02 04:34:27 0 days 05:14:06
31804   2020-02-05 00:43:00 0 days 00:10:39
31954   2020-02-05 04:28:42 0 days 03:21:05
...                     ...             ...
1502468 2020-08-30 04:26:10 0 days 03:45:10
1509758 2020-08-31 01:49:32 0 days 01:19:09
1511498 2020-08-31 06:42:31 0 days 00:05:41
1516418 2020-09-01 00:37:33 0 days 04:22:23
1516448 2020-09-01 02:37:24 0 days 01:55:04

[275 rows x 2 columns]


## Check Gaps Near Failure Dates

In [ ]:
# Define the known failure start dates from the official report
failure_dates = pd.to_datetime([
    "2020-04-18", "2020-05-29", "2020-06-05", "2020-07-15"
])

# For each failure date, check if there's a big gap within 2 days prior
for fail_date in failure_dates:
    window_start = fail_date - pd.Timedelta(days=2)
    nearby_gaps = big_gaps[
        (big_gaps['timestamp'] >= window_start) &
        (big_gaps['timestamp'] <= fail_date)
    ]
    print(f"Failure on {fail_date.date()}: {len(nearby_gaps)} big gap(s) found nearby")


Failure on 2020-04-18: 4 big gap(s) found nearby
Failure on 2020-05-29: 3 big gap(s) found nearby
Failure on 2020-06-05: 1 big gap(s) found nearby
Failure on 2020-07-15: 4 big gap(s) found nearby


#  Inspect Gap Details Near Each Failure

In [ ]:
# For each failure, show the actual gap size and timing (not just the count)
for fail_date in failure_dates:
    window_start = fail_date - pd.Timedelta(days=2)
    nearby_gaps = big_gaps[
        (big_gaps['timestamp'] >= window_start) &
        (big_gaps['timestamp'] <= fail_date)
    ]
    print(f"\n--- Failure on {fail_date.date()} ---")
    print(nearby_gaps[['timestamp', 'time_diff']])


--- Failure on 2020-04-18 ---
                 timestamp       time_diff
548411 2020-04-16 00:51:18 0 days 00:22:24
548441 2020-04-16 02:27:49 0 days 01:31:44
556450 2020-04-17 00:39:11 0 days 00:08:22
556480 2020-04-17 04:38:50 0 days 03:54:52

--- Failure on 2020-05-29 ---
                 timestamp       time_diff
822011 2020-05-27 13:53:52 0 days 00:10:07
825994 2020-05-28 03:22:24 0 days 00:05:56
826054 2020-05-28 03:37:20 0 days 00:05:10

--- Failure on 2020-06-05 ---
                 timestamp       time_diff
877509 2020-06-03 21:22:39 0 days 02:02:25

--- Failure on 2020-07-15 ---
                  timestamp       time_diff
1154831 2020-07-13 01:35:36 0 days 05:35:58
1158969 2020-07-13 14:44:15 0 days 01:45:10
1159268 2020-07-13 15:38:36 0 days 00:05:08
1164908 2020-07-14 21:28:05 0 days 14:11:28


# Conclusion: Data Coverage Around Failures

We checked whether large time gaps (>5 min) overlap with the 2-day
window before each of the 4 known failures.

- Failure 1 (2020-04-18): OK — closest gap ended >24h before failure start
- Failure 2 (2020-05-29): OK — closest gap ended >24h before failure start
- Failure 3 (2020-06-05): OK — closest gap ended ~36h before failure start
- Failure 4 (2020-07-15): ⚠️ ISSUE — a 14h11m gap ended only ~3h before failure start

Decision: MetroPT-3 alone is sufficient as our primary dataset.
Failure #4 will be excluded from any analysis that depends on the
period immediately before failure onset, due to insufficient data coverage.
We do not need a backup dataset (e.g. CMAPSS) at this stage.

In [ ]:
df.isnull().sum()

,0
Unnamed: 0,0
timestamp,0
TP2,0
TP3,0
H1,0
DV_pressure,0
Reservoirs,0
Oil_temperature,0
Motor_current,0
COMP,0


In [ ]:
df.describe()

,Unnamed: 0,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
count,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06,1.516948e+06
mean,7.584735e+06,1.367826e+00,8.984611e+00,7.568155e+00,5.595619e-02,8.985233e+00,6.264418e+01,2.050171e+00,8.369568e-01,1.606106e-01,9.198483e-01,8.326640e-01,3.420025e-03,9.914368e-01,9.041556e-01,9.371066e-01
std,4.379053e+06,3.250930e+00,6.390951e-01,3.333200e+00,3.824015e-01,6.383070e-01,6.516261e+00,2.302053e+00,3.694052e-01,3.671716e-01,2.715280e-01,3.732757e-01,5.838091e-02,9.214078e-02,2.943779e-01,2.427712e-01
min,0.000000e+00,-3.200000e-02,7.300000e-01,-3.600000e-02,-3.200000e-02,7.120000e-01,1.540000e+01,2.000000e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.792368e+06,-1.400000e-02,8.492000e+00,8.254000e+00,-2.200000e-02,8.494000e+00,5.777500e+01,4.000000e-02,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
50%,7.584735e+06,-1.200000e-02,8.960000e+00,8.784000e+00,-2.000000e-02,8.960000e+00,6.270000e+01,4.500000e-02,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
75%,1.137710e+07,-1.000000e-02,9.492000e+00,9.374000e+00,-1.800000e-02,9.492000e+00,6.725000e+01,3.807500e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
max,1.516947e+07,1.067600e+01,1.030200e+01,1.028800e+01,9.844000e+00,1.030000e+01,8.905000e+01,9.295000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


#Conclusion: Data Quality Check

- No missing values found in any column.
- Row count (1,516,948) is exactly 1/10 of the official dataset size
  (15,169,480), matching our earlier finding that the sampling interval
  is ~10s instead of the documented 1Hz. This file appears to be a
  downsampled version of the original dataset.
- Small negative values near zero in TP2, H1, and DV_pressure (e.g. -0.03)
  are consistent with normal sensor noise, not invalid readings.
- No physically impossible values (e.g. extreme negative pressure/temperature) found.

Decision: The dataset is clean enough to proceed. No row removal needed
at this stage.

In [ ]:
# Check that digital sensor columns only contain 0 or 1 (no unexpected values)
digital_cols = ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS',
                 'Pressure_switch', 'Oil_level', 'Caudal_impulses']

for col in digital_cols:
    print(col, "→", df[col].unique())

COMP → [1. 0.]
DV_eletric → [0. 1.]
Towers → [1. 0.]
MPG → [1. 0.]
LPS → [0. 1.]
Pressure_switch → [1. 0.]
Oil_level → [1. 0.]
Caudal_impulses → [1. 0.]


#  Final Conclusion: Data Understanding Complete

Summary of findings:
- 15 sensor columns confirmed matching official documentation (7 analogue + 8 digital)
- Sampling interval is ~10s (this file is a downsampled version of the original 1Hz dataset)
- No missing values in any column
- Value ranges are physically plausible; small negative values near zero
  are normal sensor noise, not errors
- Digital sensors verified to strictly contain only 0/1 values, matching documentation
- 3 of 4 known failures have sufficient sensor coverage in the period
  immediately before failure onset; failure #4 (2020-07-15) will be excluded
  from time-sensitive analysis due to a 14h11m data gap ending ~3h before it

Decision: Data Understanding phase complete. Moving to Exploratory Data
Analysis (EDA) — visualizing sensor behavior, especially around the 3
usable failure events.

 ## Minimal Cleaning Steps

In [ ]:
# Remove the auto-generated index column — it has no physical meaning
# and is not a real sensor reading
df = df.drop(columns=['Unnamed: 0'])

# Confirm the column was removed
print(df.columns.tolist())

['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']
